# 02 - Time-based split and features

In this notebook I split the data by time into train, validation and test, and then prepare the features.

**Rule:** I split first. Everything I learn from the data (averages for filling gaps, categories, scaling) comes from **train only** and is then applied to validation and test. Otherwise information from the future leaks into training and the scores look better than they really are.

| Part | Job |
|---|---|
| Train | the model learns from it |
| Validation | I tune with it: model choice, threshold, early stopping |
| Test | final score, used once at the end |

## 1. Load and merge the data

Same as in notebook 01: the two train files, joined with a left join on `TransactionID` so every transaction is kept.

In [1]:
import pandas as pd

# load the two train files (path goes up one folder from notebooks/)
df = pd.read_csv('../data/raw/train_transaction.csv')
df1 = pd.read_csv('../data/raw/train_identity.csv')

# left join keeps every transaction; device columns are NaN where there is no identity row
df_merged = pd.merge(df, df1, on='TransactionID', how='left')

# expected: (590540, 434)
df_merged.shape

(590540, 434)

## 2. Time-based split

The data covers about 26 weeks and is already in time order. I want roughly 70% train, 15% validation and 15% test, with train the earliest part and test the latest.

**My plan:**

| Part | Weeks | Why |
|---|---|---|
| Train | 0-17 | the past, the model learns from it |
| Validation | 18-21 | the next weeks, for tuning |
| Test | 22-26 | the most recent weeks, final score once |

I split on whole weeks so no week is cut in half, and it is easy to explain: "I trained on the first 18 weeks".

### Week number

This notebook needs the `week` column again. `TransactionDT` is in seconds, one week is 86,400 x 7 seconds, and `//` drops the decimals.

In [2]:
# week number of each transaction (0 to 26)
df_merged['week'] = df_merged['TransactionDT'] // (86400 * 7)

# check: should be 0 and 26
print(df_merged['week'].min(), df_merged['week'].max())

0 26


C:\Users\HP\AppData\Local\Temp\ipykernel_17080\4177173830.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_merged['week'] = df_merged['TransactionDT'] // (86400 * 7)


### Masks for the three parts

Each mask asks a question about the week for every row (True/False). The parts must not overlap: week 21 belongs to validation only, so test starts at 22.

**Watch out:** when two conditions are joined with `&`, each one needs its own brackets. Without them Python runs `&` before `>=` and builds a wrong mask without any error.

In [3]:
# a mask is True/False for every row; each one picks the weeks for one part
# train: weeks 0-17 (the past)
train_mask = df_merged['week'] <= 17

# validation: weeks 18-21; two conditions, each in its own brackets, joined with & (and)
validation_mask = (df_merged['week'] >= 18) & (df_merged['week'] <= 21)

# test: weeks 22-26 (the most recent part)
test_mask = df_merged['week'] >= 22

### Check: does every row land in exactly one part?

In [4]:
# summing True/False counts the True values = number of rows in each part
print(train_mask.sum(), validation_mask.sum(), test_mask.sum())

431125 77090 82325


**Result:**

| Part | Rows | Share |
|---|---|---|
| Train | 431,125 | 73% |
| Validation | 77,090 | 13% |
| Test | 82,325 | 14% |
| **Total** | **590,540** | 100% |

The three parts add up to exactly 590,540, so every transaction is in one part only: no gaps and no overlaps.

### Cut the data into three tables and check the fraud rate in each

The masks only point at rows (True/False). Filtering `df_merged` with a mask makes a real, separate table:

- `train` (weeks 0-17): the model learns from this table only, and everything in preprocessing (fill values, category lists, scaling) is learned from it too.
- `val` (weeks 18-21): used to compare models and choose the threshold. The model never learns from it.
- `test` (weeks 22-26): kept aside and used once at the end for the final score.

Clear names matter: if the test table had a name like `train2`, it would be easy to train on it by mistake, which is leakage.

From the weekly chart in notebook 01, the early weeks had less fraud, so train may be a bit below 3.5%.

In [5]:
# filter df_merged with each mask (no quotes: the mask is a variable, not a column name)
train = df_merged[train_mask]
val = df_merged[validation_mask]
test = df_merged[test_mask]

In [6]:
# fraud rate (%) in each part: mean of the 0/1 isFraud column x 100
# (overall rate for comparison: 3.50%)
print(train['isFraud'].mean() * 100)
print(val['isFraud'].mean() * 100)
print(test['isFraud'].mean() * 100)

3.5140620469701367
3.4310546114930602
3.4837534163376858


**Result:**

| Part | Weeks | Rows | Fraud rate |
|---|---|---|---|
| Train | 0-17 | 431,125 | 3.51% |
| Validation | 18-21 | 77,090 | 3.43% |
| Test | 22-26 | 82,325 | 3.48% |
| *All data* | | *590,540* | *3.50%* |

All three parts have almost the same fraud rate as the whole dataset. So the model learns from, is tuned on and is tested on data with the same amount of fraud, and any difference between validation and test scores later will not come from different fraud rates.

I expected train to be lower, because weeks 0-3 had little fraud. But weeks 4-10 had more fraud than average, and both are inside train, so they balance out.

## 3. Clues and answer: X and y

A model needs the data in two parts:

- **X:** the clues the model looks at (amount, product, card, device...)
- **y:** the answer it tries to predict (`isFraud`)

If `isFraud` stayed in X, the model would just read the answer and score 100%, but it would be useless in real life, where nobody knows the answer in advance. That is the most extreme kind of leakage.

Some other columns are not real clues either and have to be removed from X: `TransactionID`, `TransactionDT` and `week`.

*Why each of these would be a mistake as a clue:*

In [7]:
# y = the answer the model has to predict
y_train = train['isFraud']

In [8]:
# X = the clues: everything except the answer, the ID and the time columns
X_train = train.drop(columns=['isFraud', 'TransactionID', 'TransactionDT', 'week'])

**Why the ID and time columns are removed:**

- `TransactionID` is just a receipt number. It grows over time, so the model could memorise "IDs around 3,100,000 had more fraud", but new payments get IDs it has never seen.
- `TransactionDT` and `week` are the time of the payment. Every future payment has a later time than anything in training, so a pattern about specific times does not carry over. Time is used for the split, not as a clue.

A good clue describes the payment itself (amount, product, card, device, email) and means the same thing for a payment next month.

Next: the same X and y for validation and test.

In [9]:
# same X and y for validation and test, with the same 4 columns removed
y_val = val['isFraud']
X_val = val.drop(columns=['isFraud', 'TransactionID', 'TransactionDT', 'week'])

y_test = test['isFraud']
X_test = test.drop(columns=['isFraud', 'TransactionID', 'TransactionDT', 'week'])

# all three should have 431 columns (434 + week = 435, minus 4)
print(X_train.shape, X_val.shape, X_test.shape)

(431125, 431) (77090, 431) (82325, 431)


**Result:** `X_train` (431,125 x 431), `X_val` (77,090 x 431), `X_test` (82,325 x 431). All three have the same 431 columns (435 - 4: `df_merged` has 434 columns plus the `week` column I added), so the model sees the same clues everywhere.

## 4. Drop almost-empty columns

**My rule:** drop columns that are more than 90% missing. Not lower, because the device columns are about 76% missing and still useful: transactions with device data had 7.96% fraud against 2.10% without.

**Decided on train only.** In the EDA I found 12 such columns, but that was on the whole dataset, which includes the test weeks. Using that list would let the test period influence a training decision, so I calculate it again on `X_train`.

In [10]:
# missing % of every column, calculated on X_train only
missing_pct_train = X_train.isnull().mean() * 100

# names of the columns that are more than 90% missing (.index holds the column names)
cols_to_drop = missing_pct_train[missing_pct_train > 90].index

print(len(cols_to_drop))
print(missing_pct_train[cols_to_drop].sort_values(ascending=False))

12
id_24    99.145723
id_25    99.073587
id_08    99.070107
id_07    99.070107
id_26    99.069411
id_21    99.068948
id_22    99.068020
id_23    99.068020
id_27    99.068020
D7       93.565439
dist2    93.099681
id_18    91.889127
dtype: float64


**Result:** 12 columns are more than 90% missing in train:

| Column | Missing in train |
|---|---|
| id_24 | 99.2% |
| id_25, id_07, id_08, id_21, id_22, id_23, id_26, id_27 | 99.1% |
| D7 | 93.6% |
| dist2 | 93.1% |
| id_18 | 91.9% |

They are the same 12 columns as in the EDA, so here the decision would have been the same either way. It still matters to do it on train only, because on other data the two lists can differ.

The same `cols_to_drop` list will be removed from `X_train`, `X_val` and `X_test`, so all three keep exactly the same columns (431 - 12 = 419).

### Remove the 12 columns from all three X tables

The list was decided on train only, but it has to be removed from train, validation and test. If one table kept a column the others do not have, the model would get different clues at test time and break.

Run this cell only once: a second run would try to drop columns that are already gone and give a KeyError.

In [11]:
# drop the same 12 almost-empty columns from all three X tables
X_train = X_train.drop(columns=cols_to_drop)
X_val = X_val.drop(columns=cols_to_drop)
X_test = X_test.drop(columns=cols_to_drop)

# check: each should now have 419 columns (431 - 12)
print(X_train.shape, X_val.shape, X_test.shape)

(431125, 419) (77090, 419) (82325, 419)


**Result:** `X_train` (431,125 x 419), `X_val` (77,090 x 419), `X_test` (82,325 x 419). Same rows as before, and 12 fewer columns in each.

**Next:** sort the 419 columns into numbers and categories (text), because the pipeline treats them differently.

## 5. Numbers and categories

The pipeline treats the two kinds of columns differently: number columns get their gaps filled (and scaled for Logistic Regression), and text columns get **encoded** into numbers. So it needs two lists of column names.

`.dtypes` gives the type of every column. `select_dtypes(include='number')` keeps the number columns, and `exclude='number'` keeps everything else. In pandas 3, text columns can show up as `str` or `object`, so "not a number" is the safe way to catch all of them.

In [12]:
# how many columns of each type
print(X_train.dtypes.value_counts())

# two lists of column names for the pipeline
num_cols = X_train.select_dtypes(include='number').columns
cat_cols = X_train.select_dtypes(exclude='number').columns

print(len(num_cols), len(cat_cols))
print(list(cat_cols))

float64    389
str         29
int64        1
Name: count, dtype: int64
390 29
['ProductCD', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'M1', 'M2', 'M3', 'M4', 'M5', 'M6', 'M7', 'M8', 'M9', 'id_12', 'id_15', 'id_16', 'id_28', 'id_29', 'id_30', 'id_31', 'id_33', 'id_34', 'id_35', 'id_36', 'id_37', 'id_38', 'DeviceType', 'DeviceInfo']


**Result:** 390 number columns (389 float, 1 int) and 29 text columns, 419 in total.

The text columns are: `ProductCD`, `card4`, `card6`, `P_emaildomain`, `R_emaildomain`, `M1`-`M9`, `DeviceType`, `DeviceInfo` and 13 identity columns (`id_12`, `id_15`, `id_16`, `id_28`-`id_31`, `id_33`-`id_38`).

**Something to keep in mind:** Kaggle lists `card1`, `card2`, `card3`, `card5`, `addr1` and `addr2` as categories, but they are stored as numbers (codes like 13926), so they landed in `num_cols`. A code is not a real amount: card 13926 is not "bigger" than card 2755. For now I keep them as numbers, because `card1` alone has thousands of different values and turning each into its own column would explode the table. XGBoost can still split on them. This is a known limitation of the Logistic Regression baseline.